In [0]:

from pyspark.sql import functions as f 
from pyspark.sql.types import *


In [0]:
CATALOG         = "de_workspace26"
SCHEMA          = "freshcart_team1"
bronze_path     = "s3://s3-de-q1-26/DE-Training/freshcart-datalake-team1-db/bronze"
silver_path     = "s3://s3-de-q1-26/DE-Training/freshcart-datalake-team1-db/silver"
CHECKPOINT_BASE = "s3://s3-de-q1-26/DE-Training/freshcart-datalake-team1-db/checkpoints/silver"



In [0]:
# Read orders from Bronze. Apply transformations: drop null order_id, cast total_amount to Double, cast order_date to Date, standardise status to lowercase, add processing_date


checkpoint_path = f"{CHECKPOINT_BASE}/orders/checkpoints/"
table_name      = f"{CATALOG}.{SCHEMA}.silver_data_orders"


df_bronze_orders=df_bronze = spark.readStream \
    .format("delta") \
    .load(f"{bronze_path}/orders/")

df_silver_orders_1 = df_bronze_orders.filter(
    f.col('order_id').isNotNull()
).withColumn(
    'total_amount', f.col('total_amount').cast(DoubleType())
).withColumn(
    'order_date', f.col('order_date').cast(DateType())
).withColumn(
    'status', f.lower(f.col('status'))
).withColumn(
    'processing_date', f.current_date()
)




df_silver_orders_1=df_silver_orders_1.dropDuplicates(['order_id'])





query = (
        df_silver_orders_1.writeStream
                 .format("delta")
                 .option("checkpointLocation", checkpoint_path)
                 .option("mergeSchema", "true")
                 .option("path", f"{silver_path}/orders/")
                 .partitionBy("processing_date")
                 .outputMode("append")
                 .trigger(availableNow=True)
                 .toTable(table_name)
    )



query.awaitTermination()
print("All records processed for orders")





In [0]:
from delta.tables import DeltaTable

df_processed_silver=spark.read.format("delta").load(f"{silver_path}/orders/")
delta_table = DeltaTable.forName(spark, "freshcart_team1.silver_data_orders")

delta_table.alias("target").merge(
    df_processed_silver.alias("source"),
    "target.order_id = source.order_id"
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

In [0]:
checkpoint_path = f"{CHECKPOINT_BASE}/customers/checkpoints/"
table_name      = f"{CATALOG}.{SCHEMA}.silver_data_customers"

df_check = spark.read.format("delta").load(f"{bronze_path}/customers/")
rows_before = df_check.count()
print(f"Row count before cleaning: {rows_before}")
df_check.show(5)

df_bronze_customers = (
    spark.readStream
         .format("delta")
         .load(f"{bronze_path}/customers/")
)

df_silver_customers = (
    df_bronze_customers
    .filter(f.col("customer_id").isNotNull())
    .withColumn("city",            f.initcap(f.col("city")))
    .withColumn("processing_date", f.current_date())
)

query = (
    df_silver_customers.writeStream
                       .format("delta")
                       .outputMode("append")
                       .option("checkpointLocation", checkpoint_path)
                       .option("path", f"{silver_path}/customers/")
                       .option("mergeSchema", True)
                       .partitionBy("processing_date")
                       .trigger(availableNow=True)
                       .toTable(table_name)
)

query.awaitTermination()

df_after = spark.read.format("delta").load(f"{silver_path}/customers/")
rows_after = df_after.count()


In [0]:

print(" dif in before and after cleaning is ", rows_before-rows_after)


In [0]:
from delta.tables import DeltaTable

df_silver_customers=spark.read.format("delta").load(f"{silver_path}/customers/")
dt=DeltaTable.forName(spark,'freshcart_team1.silver_data_customers')

dt.alias('target').merge(
    df_silver_customers.alias('source'),
    'target.customer_id=source.customer_id'
).whenMatchedUpdateAll(
).whenNotMatchedInsertAll(
).execute()

In [0]:
%sql

ALTER TABLE freshcart_team1.silver_data_orders SET TBLPROPERTIES (delta.enableChangeDataFeed = true)


In [0]:
def transform_deliveries(df):
    return (
        df
        .filter(f.col("delivery_id").isNotNull())
        .withColumn("pickup_time",      f.col("pickup_time").cast(DateType()))
        .withColumn("drop_time",        f.col("drop_time").cast(DateType()))
        .withColumn("delivery_minutes", f.col("delivery_minutes").cast(IntegerType()))
        .withColumn("distance_km",      f.col("distance_km").cast(DoubleType()))
        .withColumn("rating",           f.col("rating").cast(DoubleType()))
        .withColumn("processing_date",  f.current_date())
        .dropDuplicates(["delivery_id"])
    )

def transform_products(df):
    return (
        df
        .filter(f.col("product_id").isNotNull())
        .withColumn("unit_price",       f.col("unit_price").cast(DoubleType()))
        .withColumn("category",         f.lower(f.col("category")))
        .withColumn("in_stock",         f.col("in_stock").cast(BooleanType()))
        .withColumn("processing_date",  f.current_date())
        .dropDuplicates(["product_id"])
    )

def transform_order_items(df):
    return (
        df
        .filter(f.col("item_id").isNotNull())
        .withColumn("quantity",         f.col("quantity").cast(IntegerType()))
        .withColumn("unit_price",       f.col("unit_price").cast(DoubleType()))
        .withColumn("line_total",       f.col("line_total").cast(DoubleType()))
        .withColumn("processing_date",  f.current_date())
        .dropDuplicates(["item_id"])
    )


transforms = {
    "deliveries":  transform_deliveries,
    "products":    transform_products,
    "order_items": transform_order_items,
}


queries = []

for table, transform_fn in transforms.items():

    checkpoint_path = f"{CHECKPOINT_BASE}/{table}/checkpoints/"
    table_name      = f"{CATALOG}.{SCHEMA}.silver_data_{table}"

    print(f"Processing: {table}")

  
    df_bronze = (
        spark.readStream
             .format("delta")
             .load(f"{bronze_path}/{table}/")
    )

   
    df_silver = transform_fn(df_bronze)


    query = (
        df_silver.writeStream
                 .format("delta")
                 .option("checkpointLocation", checkpoint_path)
                 .option("mergeSchema", "true")
                 .option("path", f"{silver_path}/{table}/")
                 .partitionBy("processing_date")
                 .outputMode("append")
                 .trigger(availableNow=True)
                 .toTable(table_name)
    )

    queries.append(query)

for query in queries:
    query.awaitTermination()

print("Silver ingestion complete for deliveries, products, order_items!")

In [0]:
%sql
-- # %sql
-- # S3a. Running revenue total — Using a CTE named monthly_revenue, compute
-- #      total_amount per month per city. Then use SUM() OVER (PARTITION BY city
-- #      ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
-- #      to compute a running total per city.
-- #      Show: city, month, monthly_revenue, running_total



with  monthly_revenue as (

  select city , date_format(order_date,'yyyy-MM') as month, sum(total_amount) as revenue
  from freshcart_team1.silver_orders
  group by city , date_format(order_date,'yyyy-MM')
)

select * , sum(revenue) over(PARTITION BY city
     ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) as running_total
     from monthly_revenue


In [0]:
%sql
-- S3b. Customer order frequency ranking — Using a CTE named order_counts,
--      count orders per customer. Then RANK() customers within each city
--      by order count descending. Show top 3 customers per city.
--      Show: city, customer_id, order_count, city_rank
--      Filter: WHERE city_rank <= 3


with order_counts as (

  select customer_id,city, count(*) as count_order
  from freshcart_team1.silver_orders
  group by customer_id,city
)


select *  from (

select *, rank() over(partition by city order by count_order desc) as city_rank
from order_counts

)
where city_rank<=3;



In [0]:
# %sql
# DROP TABLE IF EXISTS de_workspace26.freshcart_team1.silver_data_customers;